Data Loading

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

DATASET_DIR = "/kaggle/input/competitions/icdar-2026-circleid-writer-identification"
PRETRAINED_DIR = "/kaggle/input/datasets/cbdsigmas/tranfer-to-writer-model"

train_1 = pd.read_csv(f"{DATASET_DIR}/train.csv")
additional_path = Path(DATASET_DIR) / "additional_train.csv"
if additional_path.exists():
    train_2 = pd.read_csv(additional_path)
    train_df = pd.concat([train_1, train_2], ignore_index=True)
    del(train_2)
else:
    train_df = train_1.copy()

test_df = pd.read_csv(f"{DATASET_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATASET_DIR}/sample_submission.csv")

del(train_1)
print(f"train_df shape: {train_df.shape}")
print(f"test_df shape: {test_df.shape}")

train_df.sample(min(7, len(train_df)))


train_df shape: (40250, 4)
test_df shape: (5905, 2)


,image_id,image_path,writer_id,pen_id
11586,19564,images/19564.png,W38,4
13020,21996,images/21996.png,W40,3
5167,8715,images/08715.png,W03,3
31524,18827,images/18827.png,W41,4
11595,19577,images/19577.png,W06,3
13694,23082,images/23082.png,W50,6
32800,21947,images/21947.png,-1,3


In [2]:
if "image_path" in train_df.columns:
    train_df["image_path"] = train_df["image_path"].astype(str).map(
        lambda x: x if x.startswith("/") else f"{DATASET_DIR}/{x}"
    )
else:
    train_df["image_path"] = train_df["image_id"].astype(str).map(
        lambda x: f"{DATASET_DIR}/images/{x}.png"
    )

if "image_path" in test_df.columns:
    test_df["image_path"] = test_df["image_path"].astype(str).map(
        lambda x: x if x.startswith("/") else f"{DATASET_DIR}/{x}"
    )
else:
    test_df["image_path"] = test_df["image_id"].astype(str).map(
        lambda x: f"{DATASET_DIR}/images/{x}.png"
    )

train_df["writer_id"] = train_df["writer_id"].astype(str)
unlabeled_mask = train_df["writer_id"].eq("-1")
unlabeled_count = int(unlabeled_mask.sum())

train_df = train_df.loc[~unlabeled_mask].reset_index(drop=True)

writer_ids = sorted(train_df["writer_id"].unique())
writer_to_label = {writer_id: idx for idx, writer_id in enumerate(writer_ids)}
label_to_writer = {idx: writer_id for writer_id, idx in writer_to_label.items()}
NUM_CLASSES = len(writer_ids)

train_df["label"] = train_df["writer_id"].map(writer_to_label)
print("Dropped unlabeled rows:", unlabeled_count)
print("NUM_CLASSES:", NUM_CLASSES)
train_df[["writer_id", "label"]].head()


Dropped unlabeled rows: 5600
NUM_CLASSES: 44


,writer_id,label
0,W27,21
1,W17,15
2,W01,0
3,W17,15
4,W24,19


In [3]:
import os
import torch

XLA_AVAILABLE = False
pl = None
xm = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    XLA_AVAILABLE = True
except ImportError:
    pass

USE_TPU = XLA_AVAILABLE and (
    os.environ.get("TPU_NAME") is not None
    or os.environ.get("PJRT_DEVICE") == "TPU"
)

if USE_TPU:
    os.environ.setdefault("PJRT_DEVICE", "TPU")
    device = xm.xla_device()
    print("Using TPU:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", device)
else:
    device = torch.device("cpu")
    print("Using CPU:", device)


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


/tmp/ipykernel_74/3154709306.py:23: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
E0000 00:00:1774074844.718114      74 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Using TPU: xla:0


Train/validation split

In [4]:
from sklearn.model_selection import train_test_split

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

print(train_split_df.shape, val_split_df.shape)


(29452, 5) (5198, 5)


Dataset creation

In [5]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class CircleDataset(Dataset):
    def __init__(
        self,
        df,
        img_root,
        transform=None,
        label_map=None,
        is_test=False,
        return_path=False
    ):
        self.df = df.reset_index(drop=True)
        self.root = Path(img_root)
        self.transform = transform
        self.label_map = label_map
        self.is_test = is_test
        self.return_path = return_path

        # Auto-create label map if not provided (train mode only)
        if not self.is_test and self.label_map is None:
            unique_labels = sorted(self.df["writer_id"].unique())
            self.label_map = {k: i for i, k in enumerate(unique_labels)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Robust path handling
        img_path = self.root / row["image_path"]

        # Safe image loading (prevents crash on corrupted files)
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # fallback: blank image
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            if self.return_path:
                return image, row["image_id"], str(img_path)
            return image, row["image_id"]

        # Label encoding
        label = self.label_map[row["writer_id"]]

        if self.return_path:
            return image, label, str(img_path)

        return image, label


Image Transformation

In [6]:
from torchvision import transforms

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),

    transforms.RandomRotation(degrees=20),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.05
    ),

    transforms.ToTensor(),

    normalize,

    transforms.RandomErasing(
        p=0.2,
        scale=(0.02, 0.1),
        ratio=(0.3, 3.3)
    ),
])


val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

Loading Model

In [7]:
import torch.nn as nn
from torchvision.models import resnet18, convnext_tiny

PRETRAINED_PATHS = {
    "resnet": f"{PRETRAINED_DIR}/resnet_best.pth",
    "convnext": f"{PRETRAINED_DIR}/convnext_best.pth",
}


def load_pretrained_backbone(model, model_name):
    ckpt_path = PRETRAINED_PATHS[model_name]

    if not os.path.exists(ckpt_path):
        print(f"Missing pretrained {model_name} checkpoint: {ckpt_path}")
        return model

    checkpoint = torch.load(ckpt_path, map_location="cpu")
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]
        elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
            checkpoint = checkpoint["model"]

    checkpoint = {
        key.replace("module.", ""): value
        for key, value in checkpoint.items()
    }

    if model_name == "resnet":
        checkpoint = {
            key: value for key, value in checkpoint.items()
            if not key.startswith("fc.")
        }
    else:
        checkpoint = {
            key: value for key, value in checkpoint.items()
            if not key.startswith("classifier.2")
        }

    missing, unexpected = model.load_state_dict(checkpoint, strict=False)
    print(f"Loaded pretrained {model_name} from {ckpt_path}")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model


def build_model(model_name="resnet"):

    if model_name == "resnet":

        model = resnet18(weights="IMAGENET1K_V1")
        model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(model.fc.in_features, NUM_CLASSES)
        )

    elif model_name == "convnext":

        model = convnext_tiny(weights="IMAGENET1K_V1")
        model.classifier[2] = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
        )

    model = load_pretrained_backbone(model, model_name)
    return model


Model Training

In [8]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

IS_XLA = str(device).startswith("xla")

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.05)

        loss.backward()

        if IS_XLA:
            xm.optimizer_step(optimizer, barrier=True)
            xm.mark_step()
        else:
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [9]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)

        preds = logits.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if IS_XLA:
            xm.mark_step()

    return correct / total


In [10]:
from torch.utils.data import DataLoader

BATCH_SIZE = 96
EPOCHS_resnet = 6
EPOCHS_convnext = 4
NUM_WORKERS = 2
PIN_MEMORY = not IS_XLA
LOWER_LR = 1e-4

models = []

train_ds = CircleDataset(train_split_df, DATASET_DIR, train_tfms)
val_ds = CircleDataset(val_split_df, DATASET_DIR, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

for model_name in ["resnet", "convnext"]:

    print(f"Training {model_name}")

    model = build_model(model_name).to(device)
    best_path = os.path.join(SAVE_DIR, f"writer_{model_name}_best.pth")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LOWER_LR,
        weight_decay=1e-4
    )

    if model_name == "resnet":
        EPOCHS = EPOCHS_resnet
    else:
        EPOCHS = EPOCHS_convnext

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_acc = 0.0

    for epoch in range(EPOCHS):

        if IS_XLA:
            epoch_train_loader = pl.MpDeviceLoader(train_loader, device)
            epoch_val_loader = pl.MpDeviceLoader(val_loader, device)
        else:
            epoch_train_loader = train_loader
            epoch_val_loader = val_loader

        train_loss = train_epoch(model, epoch_train_loader, optimizer, device)

        val_acc = validate(model, epoch_val_loader, device)

        print(f"{model_name} Epoch {epoch} loss {train_loss:.4f} val_acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print(f"Saved best {model_name} to {best_path}")

        scheduler.step()

    print(f"Best {model_name} val_acc: {best_val_acc:.4f}")
    state_dict = torch.load(best_path, map_location="cpu")
    model.load_state_dict(state_dict, strict=False)  # optional if mismatch
    model.to(device)
    model.eval()
    models.append(model)


Training resnet
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

 25%|██▍       | 11.0M/44.7M [00:00<00:00, 115MB/s]

 87%|████████▋ | 39.0M/44.7M [00:00<00:00, 220MB/s]

100%|██████████| 44.7M/44.7M [00:00<00:00, 212MB/s]

...

100%|█████████▉| 305/306 [01:20<00:00,  3.97it/s]

100%|██████████| 306/306 [01:20<00:00,  3.94it/s]

100%|██████████| 306/306 [01:20<00:00,  3.80it/s]

convnext Epoch 3 loss 1.2752 val_acc 0.7880


Saved best convnext to /kaggle/working/writer_convnext_best.pth
Best convnext val_acc: 0.7880


Inference

In [11]:
test_ds = CircleDataset(test_df, DATASET_DIR, val_tfms, is_test=True)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

preds = []

for model in models:

    model.eval()

    fold_preds = []

    with torch.no_grad():

        if IS_XLA:
            inference_loader = pl.MpDeviceLoader(test_loader, device)
        else:
            inference_loader = test_loader

        for images, _ in inference_loader:

            images = images.to(device)

            probs = []

            logits = model(images)
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.flip(images, dims=[3]))
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.flip(images, dims=[2]))
            probs.append(torch.softmax(logits, 1))

            logits = model(torch.rot90(images, 1, [2, 3]))
            probs.append(torch.softmax(logits, 1))

            prob = torch.mean(torch.stack(probs), dim=0)

            fold_preds.append(prob.cpu().numpy())

            if IS_XLA:
                xm.mark_step()

    preds.append(np.concatenate(fold_preds))

preds = np.mean(preds, axis=0)
labels = preds.argmax(1)
confidences = preds.max(axis=1)
CONF_THRESHOLD = 0.35

writer_preds = [
    label_to_writer[int(label)] if confidence >= CONF_THRESHOLD else "-1"
    for label, confidence in zip(labels, confidences)
]


/tmp/ipykernel_74/3589544338.py:48: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


Submission

In [12]:
submission = sample_submission.copy()
submission["writer_id"] = writer_preds
submission.to_csv("submission.csv", index=False)

submission.head()


,image_id,writer_id
0,v2_000010c19b698fe39922ef49ff26ce16,W39
1,v2_000f8ac7d2e0ecf1d336c60fe7f34789,-1
2,v2_001a7e63cc91c92d259c2bc43572824f,-1
3,v2_001dd98a324ee7d88952f2e1d0c29674,W48
4,v2_002e3bc1d82ac5121dccf470ceed4422,-1
